In [1]:
import pandas as pd
import googlemaps
import folium
import warnings
import os
from folium.plugins import MarkerCluster

warnings.filterwarnings('ignore')

# **Conservação em Andamento**

In [2]:
data = pd.read_excel('../AliancaCentro-Rio/Aliança Centro-Rio 3.0.xlsx')
latlongs = pd.read_excel("../AliancaCentro-Rio/8. LATLONGS/data/ocorrencias.xlsx")

colunas = ['Grau de risco', 'Número do protocolo', 'Status', 'Parecer', 
           'Status 2', 'Parecer 2', 'Endereço', 'Ruas', 'Ponto de referência', 'Serviços',
           'Região', 'Categoria', 'Tipo de ocorrência', 'Agente']

colunas_2 = ['Número do protocolo', 'Latitude', 'Longitude']

data = data[colunas]
latlongs = latlongs[colunas_2]

data = data.merge(latlongs, on='Número do protocolo', how='left')

# Remove ocorrências de "Fiscalização de comércio ambulante"
data = data[data["Tipo de ocorrência"] != "Fiscalização de comércio ambulante"]
data = data[data["Tipo de ocorrência"] != "Atendimento a pessoas em situação de rua"]


In [3]:
# Em andamento na coluna Status ou Em andamento na coluna Status 2
flt = ((data['Status'] == 'Em andamento') | (data['Status 2'] == 'Em andamento'))
EmAndamento = data[flt]

# Conservação na coluna Serviços
flt1 = EmAndamento['Serviços'] == 'Conservação'

ConservacaoEmAndamento = EmAndamento[flt1] # ocorrencias atuais de conservação em andamento

ConservacaoEmAndamento.to_excel('ConservacaoEmAndamento.xlsx')

In [4]:
FechadasNaoSolucionadas = data.query("Status == 'Fechado' and Parecer != 'Solucionado' and `Status 2` != 'Em andamento' and `Parecer 2` != 'Solucionado'")

FechadasNaoSolucionadas.sort_values(by = ['Agente', 'Endereço'], inplace = True)
FechadasNaoSolucionadas.reset_index(drop = True, inplace = True)

FechadasNaoSolucionadas.to_excel('FechadasNaoSolucionadas.xlsx')

---

In [5]:
ConservacaoEmAndamento.shape

(867, 16)

In [6]:
FechadasNaoSolucionadas.shape

(122, 16)

In [7]:
tipo = ["Conservação em Andamento"] * len(ConservacaoEmAndamento)
ConservacaoEmAndamento['Tipo'] = tipo

tipo = ["Fechadas Não Solucionadas"] * len(FechadasNaoSolucionadas)
FechadasNaoSolucionadas['Tipo'] = tipo

In [8]:
# dfCompleto = pd.concat([ConservacaoEmAndamento, IluminacaoEmAndamento, FechadasNaoSolucionadas], ignore_index=True)
dfCompleto = pd.concat([ConservacaoEmAndamento, FechadasNaoSolucionadas], ignore_index=True)

In [9]:
dfCompleto.shape

(989, 17)

In [10]:
dfCompleto.drop_duplicates(subset='Número do protocolo', keep='first', inplace=True)

In [11]:
dfCompleto.shape

(980, 17)

Agora, é simplesmente gerar o mapa.

In [12]:
# Conta quantas ocorrências por Endereço (para decidir se agrupa em cluster)
dict_ = dfCompleto["Endereço"].value_counts().to_dict()

# Cria o mapa
mapa = folium.Map(location=[-22.900252, -43.178084], zoom_start=18)

# Título no mapa
title_html = """
<h3 align="center" style="font-size:20px">
  <b>Ocorrências de Conservação ou <br>
     Fechadas Não Solucionadas </b>
</h3>
"""
mapa.get_root().html.add_child(folium.Element(title_html))

# 1) Cria FeatureGroups + MarkerClusters para cada tipo
conservacao_layer = folium.FeatureGroup(name="Conservação em Andamento")
conservacao_cluster = MarkerCluster().add_to(conservacao_layer)

fechadas_layer = folium.FeatureGroup(name="Fechadas Não Solucionadas")
fechadas_cluster = MarkerCluster().add_to(fechadas_layer)

In [13]:
for i, row in dfCompleto.iterrows():
    lat = row['Latitude']
    lng = row['Longitude']
    
    # Verifica se as coordenadas são válidas
    if lat is not None and lng is not None:
        tipo = row['Tipo']
        protocolo = row['Número do protocolo']
        endereco = row['Endereço']
        
        # Monta o HTML do popup
        popup_html = f"""
        <b>TIPO:</b> {tipo}<br>
        <b>Protocolo:</b> {protocolo}<br>
        <b>Endereço:</b> {endereco}<br>
        <b>Ponto de referência:</b> {row['Ponto de referência']}<br>
        <b>Grau de risco:</b> {row['Grau de risco']}<br>
        <b>Tipo de ocorrência:</b> {row['Tipo de ocorrência']}<br>
        <b>Serviço:</b> {row['Serviços']}<br>
        <b>Região:</b> {row['Região']}<br>
        <b>Status 1:</b> {row['Status']}<br>
        <b>Parecer:</b> {row['Parecer']}<br>
        <b>Status 2:</b> {row['Status 2']}<br>
        """
        
        # Se existir foto com o nome do protocolo, adiciona ao popup
        for ext in ['.jpeg', '.jpg', '.JPEG', '.JPG']:
            if os.path.isfile(f'fotos/{protocolo}{ext}'):
                popup_html += f"<img src='fotos/{protocolo}{ext}' width='250px'/>"
                break

        popup = folium.Popup(popup_html, max_width=300, min_width=300)
        
        # Escolhe a cor e a camada (cluster) conforme o tipo
        if tipo == "Conservação em Andamento":
            color = 'blue'
            current_cluster = conservacao_cluster
        elif tipo == "Fechadas Não Solucionadas":
            color = 'red'
            current_cluster = fechadas_cluster
        else:
            # Se for outro tipo não previsto, pule ou defina um default
            continue
        
        marker = folium.Marker(
            location=[lat, lng],
            popup=popup,
            icon=folium.Icon(color=color)
        )
        
        # Verifica se há mais de 1 ocorrência no mesmo endereço
        # Normalmente, para "agrupar em cluster" independe da contagem,
        # mas se você quiser tratar diferente quando só tem 1 ocorrência, pode
        if dict_[endereco] > 1:
            marker.add_to(current_cluster)
        else:
            # Se só tem 1 ocorrência naquele endereço
            # ainda assim podemos adicionar no cluster para a camada filtrar
            marker.add_to(current_cluster)

# 3) Adiciona cada FeatureGroup (com seu cluster) ao mapa
conservacao_layer.add_to(mapa)
fechadas_layer.add_to(mapa)

# 4) Adiciona o controle de camadas
folium.LayerControl().add_to(mapa)

# Salva o mapa em um arquivo HTML (abra-o no navegador para ver o resultado)
mapa.save("index.html")

In [14]:
mapa

In [15]:
dfCompleto.shape

(980, 17)